In [2]:
import pandas as pd
import plotly.express as px
import swcol as sc

# Switch Colombia: Energy Model Input Generator

This notebook focuses on generating specific inputs for Swtich: Energy planning model according to the Colombian enviroment. While the model has multiple input requirements, this notebook centers on the ones that can vary over time. These inputs include:

- `Periods`: Defines the temporal scope of the data.
- `Timepoints`: Individual timestamps used in the model.
- `Timeseries`: Temporal variations in data.
- `Loads`: Energy demands.
- `Fuel Costs`: Variations in fuel prices.
- `Variable Capacity Factors`: Wind, Solar and Minor-Water availability for hydroelectric plants.
- `Hydro Timeseries`: Water availability for big hydroelectric plants.


These inputs are not included, you must modify them externally.
- `financials`
- `fuels`
- `load_zones`
- `gen_build_costs`
- `gen_build_predetermined`
- `gen_info`
- `Non_fuel_energy_sources`

The objective is to provide a user-friendly way to modify and generate these inputs for integration into the energy planning model.


# Initial Configuration

In this section, we define the core variables that drive the input generation process. Modify these variables to suit your specific scenario. Below are the configuration options and their descriptions:

1. **Paths**:
   - `dema_path`: Path to the source demand data (e.g., XM-API variable queries).
   - `model_path`: Path to save the generated input files.

2. **Growth Rate**:
   - `grow_rate`: Annual percentage growth in energy demand.

3. **Time**:
   - `base_year`: Starting year for the simulation, aligning with the last date in the input data.
   - `period`: Number of years between generated data points.
   - `cycles`: Total number of periods for the simulation.

4. **Load Type**:
   - `load_type`: Column from the source data used to generate loads. Must be one of the following:
     - `DemaReal_Sistema`: Real demand system.
     - `DemaCome_Sistema`: Commercial demand system.
     - `Gene_Sistema`: Generated system data.
     - `GeneIdea_Sistema`: Ideal generation system.

5. **Zone Consumption Percentage**:
   - `zone_consume_perc`: Percentage of total load assigned to specific zones.

6. **Variable Capacitors**:
   - `keymap_regenerate`: Boolean, if true will regenerate the keymap to generate variable capacity factors for uknown projects.

7. **Water Timeseries**:
   - `percentile`: Represents water plant usage as a fraction of total capacity.


In [29]:
# CONFIG: CHANGE THIS
# Paths
dema_path = '../../data/XM-API/variable_query/2022-12-01_2023-11-30/'
model_path = '../../model/inputs/'
# Grow Rate
grow_rate = 0.03
# Time
base_year = 2023
period = 1
cycles = 13
# Load Type
load_type = 'DemaReal_Sistema'
# Zone Consumption Percentage
zone_consume_perc = pd.DataFrame(
    data={
    'Zone': ['Antioquia', 'Caribe', 'Nordeste', 'Oriental', 'Surocciden'],
    'Value': [12.968812, 28.515994, 11.426312, 25.573738, 21.515143]
    })
# Variable Capacity Factors
keymap_regenerate = False
# Water Timeseries
percentile = 0.33

# Time Module

This module is responsible for generating **timepoints**, **timeseries**, and **periods** for energy modeling. These elements define the temporal structure of the simulation, ensuring the generated data aligns with the model's time-based requirements.

---



**Note:** Ensure the `base_year` aligns with the input data's timestamp to avoid inconsistencies in the timepoints.

In [30]:
timepoints = sc.time.generate(model_path, cycles, period,base_year)
timepoints['timepoint_id'] = timepoints['timepoint_id'].astype('int64')

File written: ../../model/inputs/timepoints.csv
File written: ../../model/inputs/timeseries.csv
File written: ../../model/inputs/periods.csv


## Loads

In [31]:
XM_report = sc.loads.melt_xm(dema_path)
XM_report.to_csv(dema_path+'Melted_DemaGen_Sys.csv')

loads = sc.loads.generate_by_periods(
    XM_report, load_type, base_year, grow_rate, period, cycles)

In [32]:
loads = sc.loads.generate_by_zones(loads, zone_consume_perc)

Processing Files:   0%|          | 0/113880 [00:00<?, ?it/s]

In [33]:
# Clusterize
loads = sc.loads.loads_to_timestamps(loads)
sc.loads.plot_multiple_line(
    loads,'Date', 'timestamp', 'demand_mw',
    'Zone', 'Clustered Load Curves by Zone', {'demand_mw':'MW'})

In [34]:
loads = sc.time.save_loads(model_path, loads, timepoints)
loads.head()

File written: ../../model/inputs/loads.csv


,LOAD_ZONE,timepoints,zone_demand_mw
12319,Surocciden,2496,2810.644615
12318,Oriental,2496,3340.843846
12317,Nordeste,2496,1492.683077
12316,Caribe,2496,3725.206154
12315,Antioquia,2496,1694.189231


In [35]:
loads = pd.merge(
    loads, timepoints,
    left_on='timepoints', right_on='timepoint_id', how="inner")
loads = loads[['LOAD_ZONE','timepoints','zone_demand_mw','timestamp']]
loads

,LOAD_ZONE,timepoints,zone_demand_mw,timestamp
0,Surocciden,2496,2810.644615,2035_Q4_holidays_23h
1,Oriental,2496,3340.843846,2035_Q4_holidays_23h
2,Nordeste,2496,1492.683077,2035_Q4_holidays_23h
3,Caribe,2496,3725.206154,2035_Q4_holidays_23h
4,Antioquia,2496,1694.189231,2035_Q4_holidays_23h
...,...,...,...,...
12475,Nordeste,1,963.686364,2023_Q1_labor_0h
12476,Antioquia,1,1093.779740,2023_Q1_labor_0h
12477,Oriental,1,2156.870649,2023_Q1_labor_0h
12478,Surocciden,1,1814.571039,2023_Q1_labor_0h


In [ ]:
sc.loads.plot_multiple_line(
    loads,'timepoints', 'timestamp', 'zone_demand_mw',
    'LOAD_ZONE', 'Clustered Load Curves by Zone',
    {'LOAD_ZONE':'Zones', 'zone_demand_mw':'Demand (MW)', 'timepoints':'Timepoints'})

# Variable Capacitors Module

The Variable Capacitors module is responsible for generating capacity factors for renewable energy projects such as solar, wind, and small hydroelectric plants. This module creates temporal variations in generation potential based on input data, enabling realistic simulation of renewable energy contributions.

---

## Solar and Wind Generation
### 1. Generate a Base Year

A base year is created as a reference for renewable capacity factors. This step ensures alignment with the **timepoints** generated in the previous module.

In [37]:
variable_capacity = sc.variable_capacitors.generate_base_year(
    '../../model/inputs/', keymap_regenerate, base_year, timepoints)
variable_capacity

,GENERATION_PROJECT,timepoint_id,gen_max_capacity_factor,cycle
0,E_Camelia,1,0.735667,5
1,E_Camelia,10,0.763490,5
2,E_Camelia,100,0.755272,5
3,E_Camelia,101,0.765571,5
4,E_Camelia,102,0.766534,5
...,...,...,...,...
15739,S_AutSolla,99,0.000000,0
15740,S_AutLevapan,99,0.000000,-1
15741,S_AutGragas,99,0.000000,0
15742,DST_Surocciden,99,0.000000,1


### 2. Generate Time Series

Using the base year, this function generates a time series of capacity factors for renewable energy projects, scaled over the defined cycles.

In [38]:
variable_capacity = sc.variable_capacitors.generate(
    '../../model/inputs/', variable_capacity, cycles)
variable_capacity = variable_capacity.sort_values(by=['timepoint_id'], ascending=[True])
variable_capacity.head(3)

Variable Capacity Factors saved in ../../model/inputs/variable_capacity_factors.csv


,GENERATION_PROJECT,timepoint_id,gen_max_capacity_factor
192,E_GuajiraI,1,0.842105
8640,S_AutJeans,1,0.000000
8641,S_AutPintuco,1,0.000000


In [39]:
fig = px.line(
    variable_capacity[variable_capacity['timepoint_id'] <= 192],
    x="timepoint_id", y="gen_max_capacity_factor",
    title="Capacity Factors Over Time",
    labels={"timepoint_id": "Timepoint ID", "gen_max_capacity_factor": "Max Capacity Factor"},
    color="GENERATION_PROJECT"
)
fig.update_layout(
    template="plotly_white"
)
fig.show()


## Water Variable Capacity Factors
Since we don't have data for water variable capacity factors, this module is a bit longer in comparission with the previous one, to achieve the factors we get the generation for each minor-water plan from xm and then normalize dividing by the theorical maximum capacity from gen_build_predetermined.

Finally, we concatenate this to the end of the solar and wind capacity factors.

In [59]:
import swcol as sc
# Get generarion from XM
hydro_variable_capacity = sc.hydro_capacitors.get_xm_reports()
hydro_variable_capacity.head(3)

,Values_Name,Value,Datetime
0,EL POPAL,19.85625,2023-11-26
1,LA REBUSCA,0.65040,2023-11-26
2,BAJO TULUA,9.51840,2023-11-26


In [66]:
cascada = hydro_variable_capacity[hydro_variable_capacity['Values_Name'] == 'CASCADA'].sort_values(by='Datetime')

fig = px.line(
    cascada, 
    x='Datetime', y='Value', title='Full Timeserie Cascada',
    height=9*50, width=16*50,
    template="plotly_white"
)
fig.show()


In [41]:
# Transform XM names to Switch names
hydro_variable_capacity = sc.hydro_capacitors.to_swcol_names(model_path, hydro_variable_capacity)
hydro_variable_capacity.head(3)

,Name,Value,Datetime
0,H_ElPopal,19.85625,2023-11-26
1,H_LaRebusca,0.65040,2023-11-26
2,H_TuluaBajo,9.51840,2023-11-26


In [42]:
# Clusterize data
hydro_variable_capacity = sc.hydro_capacitors.cluster(hydro_variable_capacity, base_year)
hydro_variable_capacity.head(3)

,Name,timestamp,Value
0,CALDERAS,2023_Q1_holidays_0h,7.814257
1,CALDERAS,2023_Q1_holidays_10h,8.367661
2,CALDERAS,2023_Q1_holidays_11h,9.017267


In [43]:
# Normalize diving by the maximum capacity of each plant
hydro_variable_capacity = sc.hydro_capacitors.normalize(model_path, hydro_variable_capacity)
hydro_variable_capacity.head(3)

,GENERATION_PROJECT,timepoint_id,gen_max_capacity_factor
0,CALDERAS,1,0.426395
1,CALDERAS,2,0.424233
2,CALDERAS,3,0.417611


In [44]:
# Generate the upcoming years
hydro_variable_capacity = sc.hydro_capacitors.generate(model_path, hydro_variable_capacity, period, cycles)
hydro_variable_capacity.head(3)

Hydro Variable Capacitors concatenated with ../../model/inputs/variable_capacity_factors.csv


,GENERATION_PROJECT,timepoint_id,gen_max_capacity_factor
0,E_GuajiraI,1,0.842105
1,E_GuajiraI,10,0.798857
2,E_GuajiraI,100,0.772581


In [53]:
import plotly.express as px
import pandas as pd
plot_hydro_cap = hydro_variable_capacity[hydro_variable_capacity['GENERATION_PROJECT'] == 'H_CascadaAnt']
plot_hydro_cap = plot_hydro_cap[plot_hydro_cap['timepoint_id'] <= 192].sort_values(by='timepoint_id')

plot_hydro_cap = pd.merge(
    plot_hydro_cap, timepoints,
    on='timepoint_id', how="inner")
fig = px.line(
    plot_hydro_cap, 
    x='timestamp', y='gen_max_capacity_factor', title='Capacidad por H_CascadaAnt',
    height=9*50, width=16*50,
    template="plotly_white"
)
fig.show()

## Fuel Costs

In [46]:
fuel_costs = sc.fuel_costs.generate(model_path, base_year, cycles, period)
fuel_costs.head()

Fuel cost data saved to ../../model/inputs/fuel_cost.csv


,load_zone,fuel,period,fuel_cost
0,Caribe,GASIMPOR,2023,12.716956
28,Caribe,GASNACIO,2023,7.231772
56,Nordeste,GASNACIO,2023,4.615200
84,Antioquia,GASNACIO,2023,7.231772
112,Caribe,CARBON,2023,4.133101


## Hydro Timeseries

In [47]:
import swcol as sc
generation_xm = sc.hydro_timeseries.get_xm_reports()
generation_xm

,Values_Name,Value,Datetime
0,EL POPAL,19.85625,2023-11-26 00:00:00
1,LA REBUSCA,0.65040,2023-11-26 00:00:00
2,BAJO TULUA,9.51840,2023-11-26 00:00:00
3,AUTOG ARGOS EL CAIRO,6.60891,2023-11-26 00:00:00
4,LA FRISOLERA,0.28129,2023-11-26 00:00:00
...,...,...,...
3397723,TUNJITA,NaN,2020-12-30 23:00:00
3397724,UNION,0.41976,2020-12-30 23:00:00
3397725,URRA,85.62314,2020-12-30 23:00:00
3397726,LA VUELTA,4.21300,2020-12-30 23:00:00


In [48]:
generation = sc.hydro_timeseries.to_swcol_names(model_path, generation_xm, base_year)
generation

,Values_Name,Value,Datetime,Name,Best Match
0,ALBAN,255.02500,2023-11-26 00:00:00,H_ALBAN,ALBAN
1,BETANIA,59.98191,2023-11-26 00:00:00,BETANIA,BETANIA
2,CHIVOR,9.78328,2023-11-26 00:00:00,CHIVOR,CHIVOR
3,CARLOS LLERAS,57.92735,2023-11-26 00:00:00,C_LLERAS_R,CARLOS LLERAS
4,CALIMA,NaN,2023-11-26 00:00:00,CALIMA,CALIMA
...,...,...,...,...,...
716107,SAN MIGUEL,32.09981,2020-12-30 23:00:00,SNMIGUEL,SAN MIGUEL
716108,SAN CARLOS,863.77808,2020-12-30 23:00:00,SAN CARLOS,SAN CARLOS
716109,SAN FRANCISCO,0.00004,2020-12-30 23:00:00,SANFRANCISCO,SAN FRANCISCO
716110,SOGAMOSO,473.39247,2020-12-30 23:00:00,SOGAMOSO,SOGAMOSO


In [49]:
generation = sc.hydro_timeseries.cluster(generation, base_year, percentile)
generation

,hydro_project,timeseries,hydro_avg_flow_mw,hydro_min_flow_mw
0,AMOYA,2023_Q1_holidays,35.503285,0.53926
1,AMOYA,2023_Q1_labor,33.867372,0.00585
2,AMOYA,2023_Q2_holidays,49.183004,0.02993
3,AMOYA,2023_Q2_labor,46.842842,0.00303
4,AMOYA,2023_Q3_holidays,59.213659,1.40060
...,...,...,...,...
235,URRA,2023_Q2_labor,73.420750,65.49328
236,URRA,2023_Q3_holidays,228.589459,74.30389
237,URRA,2023_Q3_labor,229.616221,74.20494
238,URRA,2023_Q4_holidays,157.002640,76.51127


In [50]:
generation = sc.hydro_timeseries.generate(
    model_path,generation, period, cycles, base_year)
generation

Hydro Timeseries saved in ../../model/inputs/hydro_timeseries.csv


,hydro_project,timeseries,hydro_avg_flow_mw,hydro_min_flow_mw
0,AMOYA,2023_Q1_holidays,35.503285,0.53926
1,AMOYA,2023_Q1_labor,33.867372,0.00585
2,AMOYA,2023_Q2_holidays,49.183004,0.02993
3,AMOYA,2023_Q2_labor,46.842842,0.00303
4,AMOYA,2023_Q3_holidays,59.213659,1.40060
...,...,...,...,...
3115,URRA,2035_Q2_labor,73.420750,65.49328
3116,URRA,2035_Q3_holidays,228.589459,74.30389
3117,URRA,2035_Q3_labor,229.616221,74.20494
3118,URRA,2035_Q4_holidays,157.002640,76.51127
